# 02 · Bronce
Auto Loader sobre `landing/csv` → `bronze.ais_raw`. Todo como `string` (crudo inmutable), con archivo de origen y hora de ingesta. El checkpoint garantiza que cada archivo se procesa una sola vez.

In [0]:
dbutils.widgets.text("catalog", "oceanwatch_e2")
catalog = dbutils.widgets.get("catalog")
print("catalog:", catalog)

dbutils.widgets.dropdown("verificar", "no", ["si", "no"])
verificar = dbutils.widgets.get("verificar") == "si"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

src = f"/Volumes/{catalog}/landing/csv"
chk = f"/Volumes/{catalog}/ops/checkpoints/bronze_ais"
tbl = f"{catalog}.bronze.ais_raw"

# Esquema desde el header real del archivo
primer_csv = sorted(f.path for f in dbutils.fs.ls(src) if f.name.endswith(".csv"))[0]
with open(primer_csv.replace("dbfs:", "")) as fh:
    header = fh.readline().strip().split(",")
print("columnas:", header)
schema = StructType([StructField(c, StringType(), True) for c in header])

antes = spark.table(tbl).count() if spark.catalog.tableExists(tbl) else 0

q = (spark.readStream.format("cloudFiles")
     .option("cloudFiles.format", "csv")
     .option("header", "true")
     .option("rescuedDataColumn", "_rescued_data")
     .schema(schema)
     .load(src)
     .withColumn("_source_file", F.col("_metadata.file_path"))
     .withColumn("_ingested_at", F.current_timestamp())
     .writeStream
     .option("checkpointLocation", chk)
     .trigger(availableNow=True)
     .toTable(tbl))
q.awaitTermination()

despues = spark.table(tbl).count()
print(f"antes: {antes:,} | después: {despues:,} | nuevas: {despues - antes:,}")

In [0]:
if verificar:
    def show(df, n=50):
        print(df.limit(n).toPandas().to_string(index=False))

    b = spark.table(tbl)
    print("total filas:", b.count())
    show(b.groupBy("_source_file").count().orderBy("_source_file"))
    print("filas con _rescued_data:", b.filter(F.col("_rescued_data").isNotNull()).count())
else:
    print("verificación omitida (verificar = no)")

In [0]:
if verificar:
    crudo = (spark.read.text(f"/Volumes/{catalog}/landing/csv/*.csv")
             .select(F.regexp_replace(F.col("_metadata.file_path"), "^dbfs:", "").alias("_source_file"))
             .groupBy("_source_file").count()
             .withColumn("lineas_csv", F.col("count") - 1).drop("count"))

    conc = (spark.table(tbl).groupBy("_source_file").count()
            .withColumnRenamed("count", "filas_bronze")
            .join(crudo, "_source_file", "full")
            .withColumn("diferencia", F.col("filas_bronze") - F.col("lineas_csv"))
            .orderBy("_source_file"))
    show(conc)
else:
    print("conciliación omitida (verificar = no)")